# Memory x RL Interaction: Pilot Experiment

**Question**: When an evolving strategy playbook co-adapts with GRPO weight updates,
does co-evolution help? And do weights absorb the playbook?

**Conditions**:
- B: GRPO-only (no playbook)
- C: GRPO + frozen playbook
- D: GRPO + evolving playbook (ACE curation between epochs)
- C-abs: C's trained model, playbook removed at eval
- D-abs: D's trained model, playbook removed at eval

**Mini pilot**: 1 seed, 3 epochs, 30 training problems, eval on AIME 2025 (30 problems)

**Runtime**: Colab A100 (40GB works with reduced batch; 80GB comfortable). ~2 hours end-to-end.

## 1. Setup & Dependencies

In [ ]:
%%capture
!pip install trl==0.27.2 vllm==0.10.2 transformers==4.57.3 peft==0.18.1 \
    accelerate==1.12.0 datasets scipy openai matplotlib

In [ ]:
import os, sys, json, time, random, gc, signal, subprocess, requests
import numpy as np
import torch
import matplotlib.pyplot as plt
from dataclasses import dataclass
from pathlib import Path

# GPU memory optimizations — set BEFORE any CUDA allocations
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Clone repo (if on Colab)
REPO_ROOT = "/content/grounded"
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/YOUR_REPO/grounded.git {REPO_ROOT}
sys.path.insert(0, os.path.join(REPO_ROOT, "experiments/memory_rl"))

from lib.config import ExperimentConfig
from lib.data import load_math500_l4l5, load_aime_2025
from lib.answer_parsing import parse_answer, check_answer, strip_think_blocks
from lib.playbook import (
    Playbook, NullPlaybook, StaticPlaybook, ActivePlaybook,
    make_initial_playbook, Bullet,
)
from lib.rewards import majority_vote_reward, ace_reward_fn
from lib.collapse_detector import CollapseDetector
from lib.analysis import bootstrap_ci, mcnemar_test, cohens_g

# Pilot config (reduced from full experiment)
SEED = 42
N_TRAIN = 30          # subset of MATH-500 L4-5 (full: 200)
N_GRPO_EPOCHS = 3     # full: 20
NUM_GENERATIONS = 8   # full: 16
MAX_COMPLETION_LENGTH = 3072  # full: 4096
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
VLLM_PORT = 8000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def log_gpu_memory(label=""):
    """Log current GPU memory usage."""
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_mem / 1e9
        print(f"  GPU [{label}]: {alloc:.1f}GB allocated, {reserved:.1f}GB reserved, {total:.1f}GB total")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"TF32: {torch.backends.cuda.matmul.allow_tf32}")
print(f"CUDA alloc config: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'default')}")
log_gpu_memory("startup")

## 2. Data Loading

In [ ]:
all_train = load_math500_l4l5()
train_problems = random.sample(all_train, min(N_TRAIN, len(all_train)))
eval_problems = load_aime_2025()

print(f"Training: {len(train_problems)} problems (from {len(all_train)} MATH-500 L4-5)")
print(f"Eval: {len(eval_problems)} AIME 2025 problems")
print(f"\nSample training problem:")
print(f"  {train_problems[0]['problem'][:200]}...")
print(f"  Answer: {train_problems[0]['answer']}")

In [ ]:
# Quick sanity check on answer parsing
tests = [
    ("\\boxed{42}", "42"),
    ("<think>reasoning</think>The answer is \\boxed{7}", "7"),
    ("#### 100", "100"),
]
for raw, expected in tests:
    result = parse_answer(raw)
    assert result == expected, f"FAIL: {raw} -> {result}, expected {expected}"
print("Answer parsing: OK")

## 3. Baseline Evaluation

Evaluate the base model (no LoRA, no playbook) on AIME 2025 to establish the pre-training baseline.

In [ ]:
# Load vLLM engine directly (no HTTP server overhead)
# This uses the same pattern as eval_modal.py and lib/evaluation.py
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

print("Initializing vLLM engine for baseline eval...")
baseline_llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    gpu_memory_utilization=0.90,
    max_model_len=4096,
    max_num_seqs=512,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
    enforce_eager=True,
    trust_remote_code=True,
    seed=SEED,
)
print("vLLM engine ready.")

In [ ]:
# Baseline eval on AIME 2025 — BATCHED via vLLM Python API
# All 30 problems are sent as a single batch to llm.generate(), which
# exploits vLLM's continuous batching scheduler for ~10-30x speedup
# over the previous sequential HTTP approach.

# DeepSeek-R1 stop tokens
STOP_TOKENS = ["<|endoftext|>", "<\uff5cend\u2581of\u2581sentence\uff5c>"]

def format_eval_prompt(problem: str, playbook_context: str = "") -> str:
    """Format problem as chat-templated prompt for vLLM direct generation."""
    system = ("You are an expert math competition solver. "
              "Solve step-by-step. Put final answer in \\boxed{}.\n" + playbook_context)
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Solve:\n\n{problem}"},
    ]
    return eval_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def batch_evaluate(llm, problems, playbook_context="", n=1, temperature=0.7):
    """Evaluate all problems in a single batched vLLM call.
    
    Returns list of result dicts with 'id', 'correct', 'predicted' keys.
    """
    # Build ALL prompts at once
    prompts = [format_eval_prompt(p["problem"], playbook_context) for p in problems]
    
    sampling_params = SamplingParams(
        n=n,
        temperature=max(temperature, 0.01),
        max_tokens=MAX_COMPLETION_LENGTH,
        stop=STOP_TOKENS,
    )
    
    # Single batched call — vLLM processes all prompts together
    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params, use_tqdm=True)
    gen_time = time.time() - t0
    print(f"  Generation: {len(prompts)} prompts in {gen_time:.1f}s "
          f"({gen_time/len(prompts):.1f}s/prompt effective)")
    
    results = []
    for problem, output in zip(problems, outputs):
        # Take first completion (n=1 for baseline)
        text = output.outputs[0].text
        answer = parse_answer(text)
        correct = check_answer(answer, problem["answer"])
        results.append({
            "id": problem["id"],
            "correct": correct,
            "predicted": answer,
        })
        print(f"  {problem['id']}: {'Y' if correct else 'N'} "
              f"(predicted={answer}, gold={problem['answer']})")
    
    return results

print("Running baseline eval on AIME 2025 (batched)...")
baseline_results = batch_evaluate(baseline_llm, eval_problems, n=1)
baseline_acc = sum(r["correct"] for r in baseline_results) / len(baseline_results)
print(f"\nBaseline accuracy: {baseline_acc:.1%} "
      f"({sum(r['correct'] for r in baseline_results)}/{len(baseline_results)})")

In [ ]:
# Destroy vLLM engine to free GPU for GRPO training
from vllm.distributed.parallel_state import destroy_model_parallel

del baseline_llm
destroy_model_parallel()
torch.cuda.synchronize()
gc.collect()
torch.cuda.empty_cache()
print("vLLM engine destroyed. GPU freed for GRPO training.")

## 4. Static Playbook Generation

Generate Condition C's frozen playbook via 3 ACE curation episodes using the Kimi API.

In [ ]:
# Set Kimi API key (Moonshot servers - no local GPU needed)
os.environ["KIMI_API_KEY"] = "YOUR_KEY"  # TODO: set this before running

from lib.kimi_client import kimi_chat
from lib.reflector import batch_reflect
from lib.curator import curate as kimi_curate, rule_based_curate

static_pb = make_initial_playbook()
N_ACE_EPISODES = 3
PROBLEMS_PER_EPISODE = 10

for episode in range(N_ACE_EPISODES):
    print(f"\n--- ACE Episode {episode+1}/{N_ACE_EPISODES} ---")
    sample = random.sample(train_problems, PROBLEMS_PER_EPISODE)

    # Build reflect items (use ground truth as stand-in for model outputs)
    items = [{
        "problem": p["problem"],
        "solution": f"Step-by-step solution: \\boxed{{{p['answer']}}}",
        "is_correct": True,
        "bullets_used": [b.id for b in static_pb.bullets[:2]] if static_pb.bullets else [],
    } for p in sample]

    try:
        reflections = batch_reflect(items, static_pb.to_str())
        for (reflection, tags), p in zip(reflections, sample):
            for bid, tag in tags.items():
                static_pb.tag(bid, tag)
            ops = kimi_curate(static_pb, p["problem"], reflection, max_bullets=20)
            ap = ActivePlaybook(static_pb)
            ap.apply_ops(ops, max_bullets=20)
            static_pb = ap.playbook
    except Exception as e:
        print(f"Kimi API failed ({e}), using rule-based fallback")
        rule_based_curate(static_pb, items, max_bullets=20)

    print(f"  Playbook: {static_pb.size} bullets, entropy={static_pb.entropy():.2f}")

print(f"\nFinal static playbook ({static_pb.size} bullets):")
print(static_pb.to_str())

In [ ]:
# Freeze and save static playbook
static_playbook = StaticPlaybook(static_pb)
static_pb_path = "static_playbook.json"
with open(static_pb_path, "w") as f:
    json.dump(static_pb.snapshot(), f, indent=2)
print(f"Saved static playbook to {static_pb_path}")
print(f"Context length: {len(static_playbook.get_context())} chars")

## 5. Mini GRPO Training

Run 3 GRPO epochs for conditions B, C, D on 30 training problems.
Uses TRL's GRPOTrainer with vLLM colocate mode (LoRA rank 64).
~15-30 min per condition on A100.

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from transformers import AutoTokenizer, TrainerState

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompts(problems, playbook_context=""):
    """Build chat-templated prompts for GRPO training."""
    system = ("You are an expert math competition solver. "
              "Solve step-by-step. Put final answer in \\boxed{}.\n" + playbook_context)
    prompts = []
    for p in problems:
        msgs = [{"role": "system", "content": system},
                {"role": "user", "content": f"Solve:\n\n{p['problem']}"}]
        prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    return prompts

peft_config = LoraConfig(
    r=64, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM", bias="none", use_rslora=True,
)

def make_grpo_config(output_dir):
    # CRITICAL: TRL requires (num_processes * per_device_train_batch_size) % num_generations == 0
    # With 1 GPU: per_device_train_batch_size must be a multiple of num_generations.
    # Setting to NUM_GENERATIONS (8) is the minimum valid value.
    return GRPOConfig(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=NUM_GENERATIONS,  # Must be multiple of num_generations
        gradient_accumulation_steps=1,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=5e-6,
        max_grad_norm=0.5,
        bf16=True, logging_steps=1, save_strategy="no",
        num_generations=NUM_GENERATIONS,
        generation_batch_size=NUM_GENERATIONS,
        max_completion_length=MAX_COMPLETION_LENGTH,
        temperature=1.0, top_p=0.95,
        use_vllm=True, vllm_mode="colocate",
        vllm_gpu_memory_utilization=0.4,
        vllm_enable_sleep_mode=True,
        beta=0.01,
        report_to="none",
    )

# Shared completion collector for CollapseDetector (reward fns append here)
_epoch_completions = []

def _collecting_majority_vote_reward(prompts, completions, **kwargs):
    """majority_vote_reward wrapper that also collects completions for collapse detection."""
    _epoch_completions.extend(completions)
    return majority_vote_reward(prompts, completions, **kwargs)

def train_condition(condition, reward_fn, playbook_mgr, n_epochs=N_GRPO_EPOCHS):
    """Train one condition. Returns (trainer, epoch_metrics, collapse_history)."""
    output_dir = f"outputs/{condition}"
    os.makedirs(output_dir, exist_ok=True)

    prompts = build_prompts(train_problems, playbook_mgr.get_context())
    grpo_config = make_grpo_config(output_dir)

    trainer = GRPOTrainer(
        model=MODEL_NAME,
        reward_funcs=reward_fn,
        args=grpo_config,
        train_dataset=Dataset.from_dict({"prompt": prompts}),
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    epoch_metrics = []
    detector = CollapseDetector(window_size=3, entropy_threshold=0.1)

    for epoch in range(n_epochs):
        log_gpu_memory(f"{condition} epoch {epoch} pre-train")
        t0 = time.time()

        # Clear completion collector
        _epoch_completions.clear()

        # Rebuild prompts if playbook evolves (condition D)
        if hasattr(playbook_mgr, 'playbook'):
            prompts = build_prompts(train_problems, playbook_mgr.get_context())
            trainer.train_dataset = Dataset.from_dict({"prompt": prompts})

        trainer.state = TrainerState()
        trainer.lr_scheduler = None
        trainer.train()

        # Record epoch in CollapseDetector using collected completions
        if _epoch_completions:
            record = detector.record_epoch(_epoch_completions, epoch=epoch)
            print(f"    Collapse detector: entropy={record.entropy:.3f}, "
                  f"unique={record.unique_answers}/{record.total_completions}")

        dt = time.time() - t0
        pb_size = playbook_mgr.playbook.size if hasattr(playbook_mgr, 'playbook') else 0
        epoch_metrics.append({
            "epoch": epoch, "time_s": dt, "pb_size": pb_size,
            "entropy": detector.latest_entropy(),
            "collapsed": detector.is_collapsed(),
        })
        log_gpu_memory(f"{condition} epoch {epoch} post-train")
        print(f"  Epoch {epoch+1}/{n_epochs}: pb_size={pb_size}, "
              f"entropy={detector.latest_entropy():.3f}, time={dt:.0f}s")

        if detector.is_collapsed():
            print(f"  EARLY STOP: collapse detected at epoch {epoch}")
            break

    # Save adapter
    adapter_path = f"{output_dir}/adapter"
    trainer.save_model(adapter_path)
    print(f"  Saved adapter to {adapter_path}")

    return trainer, epoch_metrics, detector.get_history()

print("Training infrastructure ready.")
print(f"  per_device_train_batch_size = {NUM_GENERATIONS} (= num_generations)")
print(f"  gradient_accumulation_steps = 1")
print(f"  effective_batch_size = {NUM_GENERATIONS} prompts/step")

In [ ]:
print("=" * 60)
print("Training Condition B: GRPO-only (no playbook)")
print("=" * 60)
log_gpu_memory("pre-B")
t0 = time.time()
b_trainer, b_metrics, b_collapse = train_condition(
    "B", _collecting_majority_vote_reward, NullPlaybook()
)
print(f"Condition B done in {time.time()-t0:.0f}s")

# Free memory between conditions to avoid leaks
del b_trainer; gc.collect(); torch.cuda.empty_cache()
log_gpu_memory("post-B cleanup")

In [ ]:
print("=" * 60)
print("Training Condition C: GRPO + frozen playbook")
print("=" * 60)
log_gpu_memory("pre-C")
t0 = time.time()
c_trainer, c_metrics, c_collapse = train_condition(
    "C", _collecting_majority_vote_reward, static_playbook
)
print(f"Condition C done in {time.time()-t0:.0f}s")

del c_trainer; gc.collect(); torch.cuda.empty_cache()
log_gpu_memory("post-C cleanup")

In [ ]:
print("=" * 60)
print("Training Condition D: GRPO + evolving playbook (ACE curation)")
print("=" * 60)
log_gpu_memory("pre-D")
t0 = time.time()

active_pb = ActivePlaybook()

# Build problem lookup keyed on RAW problem text.
# ace_reward_fn uses substring matching to find the problem inside the
# full chat-templated prompt that GRPOTrainer passes.
problem_lookup = {}
for p in train_problems:
    problem_lookup[p["problem"]] = p

ace_state = {
    "playbook_mgr": active_pb,
    "pending_curate": [],
    "problem_lookup": problem_lookup,
    "episode_stats": [],
}

def d_reward_fn(prompts, completions, **kwargs):
    """ACE reward with completion collection for collapse detection."""
    _epoch_completions.extend(completions)
    return ace_reward_fn(prompts, completions, state=ace_state, **kwargs)

d_trainer, d_metrics, d_collapse = train_condition("D", d_reward_fn, active_pb)

# Deferred curation after training
if ace_state["pending_curate"]:
    n_curate = min(10, len(ace_state["pending_curate"]))
    print(f"  Running deferred ACE curation on {n_curate} items...")
    try:
        items = ace_state["pending_curate"][:n_curate]
        reflections = batch_reflect(items, active_pb.playbook.to_str())
        for (reflection, tags), item in zip(reflections, items):
            active_pb.apply_tags(tags)
            ops = kimi_curate(active_pb.playbook, item["problem"], reflection, max_bullets=20)
            active_pb.apply_ops(ops, max_bullets=20)
    except Exception as e:
        print(f"  Kimi curation failed ({e}), using rule-based")
        rule_based_curate(active_pb.playbook, ace_state["pending_curate"], max_bullets=20)

print(f"Condition D done in {time.time()-t0:.0f}s")
print(f"Final playbook: {active_pb.playbook.size} bullets")
print(f"ACE episodes recorded: {len(ace_state['episode_stats'])}")

del d_trainer; gc.collect(); torch.cuda.empty_cache()
log_gpu_memory("post-D cleanup")

## 6. Evaluation -- All Conditions + Absorption

Merge each unique LoRA adapter ONCE, load vLLM engine ONCE per merged model,
and run multiple eval passes (with/without playbook) on the same engine.

**Optimization**: C and C-abs share the same merged weights (differ only in system
prompt). Same for D and D-abs. This avoids 2 redundant LoRA merges and 2
redundant vLLM engine loads (~8-16 min saved).

| Engine Load | Conditions Evaluated |
|-------------|---------------------|
| B/merged    | B                   |
| C/merged    | C, C-abs            |
| D/merged    | D, D-abs            |

In [ ]:
from lib.evaluation import merge_lora_checkpoint

# ---------------------------------------------------------------------------
# Evaluation helpers: separate merge / engine / eval / cleanup stages
# ---------------------------------------------------------------------------

def merge_adapter(adapter_path, label):
    """Merge LoRA adapter into full model. Reuse if already merged."""
    merged_path = f"outputs/{label}/merged"
    if os.path.exists(merged_path):
        print(f"  {label}: already merged, reusing {merged_path}")
        return merged_path
    merge_lora_checkpoint(MODEL_NAME, adapter_path, merged_path)
    return merged_path

def load_eval_engine(model_path, label=""):
    """Load a vLLM engine for evaluation (no HTTP server overhead)."""
    log_gpu_memory(f"pre-load {label}")
    print(f"  Loading vLLM engine for {label} ({model_path})...")
    t0 = time.time()
    engine = LLM(
        model=model_path,
        dtype="bfloat16",
        gpu_memory_utilization=0.90,
        max_model_len=4096,
        max_num_seqs=512,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
        enforce_eager=True,
        seed=SEED,
    )
    print(f"  vLLM engine ready for {label} in {time.time()-t0:.0f}s")
    log_gpu_memory(f"post-load {label}")
    return engine

def destroy_eval_engine(engine, label=""):
    """Destroy vLLM engine and free all GPU memory."""
    del engine
    destroy_model_parallel()
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()
    log_gpu_memory(f"post-destroy {label}")

def evaluate_group(adapter_path, eval_passes, group_label):
    """Merge ONCE, load engine ONCE, run multiple eval passes, destroy.
    
    Args:
        adapter_path: Path to LoRA adapter directory
        eval_passes: List of (label, playbook_context) tuples to evaluate
        group_label: Label for the merged model (used for merge path)
    
    Returns:
        Dict mapping label -> results list
    """
    # Step 1: Merge adapter (once per group)
    merged_path = merge_adapter(adapter_path, group_label)
    
    # Step 2: Load engine (once per group)
    engine = load_eval_engine(merged_path, group_label)
    
    # Step 3: Run all eval passes on the same engine
    group_results = {}
    for label, playbook_context in eval_passes:
        print(f"\n  --- Evaluating {label} ---")
        results = batch_evaluate(
            engine, eval_problems,
            playbook_context=playbook_context,
            n=1, temperature=0.01,
        )
        acc = sum(r["correct"] for r in results) / len(results)
        print(f"  {label}: {acc:.1%} ({sum(r['correct'] for r in results)}/{len(results)})")
        group_results[label] = results
    
    # Step 4: Destroy engine (once per group)
    destroy_eval_engine(engine, group_label)
    
    return group_results

print("Evaluation infrastructure ready.")

In [ ]:
print("=" * 60)
print("Evaluating all conditions on AIME 2025")
print("=" * 60)
print("Grouped by merged model to minimize vLLM restarts:")
print("  Group 1: B        (1 merge, 1 engine load, 1 eval pass)")
print("  Group 2: C + C-abs (1 merge, 1 engine load, 2 eval passes)")
print("  Group 3: D + D-abs (1 merge, 1 engine load, 2 eval passes)")
print()

eval_all = {}
t_eval_start = time.time()

# --- Group 1: B (unique model, single eval) ---
print("--- Group 1: Condition B ---")
b_results = evaluate_group(
    adapter_path="outputs/B/adapter",
    eval_passes=[("B", "")],
    group_label="B",
)
eval_all.update(b_results)

# --- Group 2: C + C-abs (SAME merged model, playbook vs no-playbook) ---
print("\n--- Group 2: Conditions C + C-abs ---")
c_results = evaluate_group(
    adapter_path="outputs/C/adapter",
    eval_passes=[
        ("C", static_playbook.get_context()),   # With playbook
        ("C-abs", ""),                            # Without playbook (absorption)
    ],
    group_label="C",
)
eval_all.update(c_results)

# --- Group 3: D + D-abs (SAME merged model, playbook vs no-playbook) ---
print("\n--- Group 3: Conditions D + D-abs ---")
d_results = evaluate_group(
    adapter_path="outputs/D/adapter",
    eval_passes=[
        ("D", active_pb.playbook.to_str()),      # With playbook
        ("D-abs", ""),                             # Without playbook (absorption)
    ],
    group_label="D",
)
eval_all.update(d_results)

t_eval_total = time.time() - t_eval_start
print(f"\nAll evaluations complete in {t_eval_total:.0f}s")
print(f"  vLLM engine loads: 3 (was 5)")
print(f"  LoRA merges: 3 (was 5)")

In [ ]:
print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

LABELS = {
    "B": "B (GRPO-only)",
    "C": "C (Static PB)",
    "C-abs": "C-abs (Absorbed)",
    "D": "D (Active PB)",
    "D-abs": "D-abs (Absorbed)",
}

for cond in ["B", "C", "C-abs", "D", "D-abs"]:
    r = eval_all[cond]
    acc = sum(x["correct"] for x in r) / len(r)
    outcomes = [x["correct"] for x in r]
    mean, lo, hi = bootstrap_ci(outcomes)
    print(f"  {LABELS[cond]:25s}  {acc:.1%}  95% CI [{lo:.1%}, {hi:.1%}]")

## 7. Statistical Tests

In [ ]:
print("Pairwise Comparisons (McNemar's test)")
print("-" * 70)

comparisons = [
    ("D", "B", "D vs B (co-evolution helps?)"),
    ("C", "B", "C vs B (static context helps?)"),
    ("D", "C", "D vs C (evolution matters?)"),
    ("D-abs", "B", "D-abs vs B (absorption?)"),
    ("D-abs", "C-abs", "D-abs vs C-abs (co-evol absorbs more?)"),
]

for a, b, label in comparisons:
    oa = [x["correct"] for x in eval_all[a]]
    ob = [x["correct"] for x in eval_all[b]]
    p, n01, n10 = mcnemar_test(oa, ob)
    g = cohens_g(oa, ob)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {label:40s}  p={p:.4f} {sig:>4s}  g={g:.3f}")

In [ ]:
print("\n" + "=" * 60)
print("ABSORPTION TEST")
print("=" * 60)

d_acc = sum(x["correct"] for x in eval_all["D"]) / len(eval_all["D"])
dabs_acc = sum(x["correct"] for x in eval_all["D-abs"]) / len(eval_all["D-abs"])
b_acc = sum(x["correct"] for x in eval_all["B"]) / len(eval_all["B"])

print(f"  D (with playbook):    {d_acc:.1%}")
print(f"  D-abs (no playbook):  {dabs_acc:.1%}")
print(f"  B (never had PB):     {b_acc:.1%}")
print(f"  Absorption gap (D - D-abs): {d_acc - dabs_acc:+.1%}")

if d_acc - dabs_acc < 0.05:
    print("  -> Weights ABSORBED the playbook (removal doesn't hurt)")
elif dabs_acc > b_acc + 0.03:
    print("  -> PARTIAL absorption (some internalization, playbook still helps)")
else:
    print("  -> NO absorption (playbook was scaffolding only)")

## 8. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fig 1: Accuracy bars
conditions = ["B", "C", "C-abs", "D", "D-abs"]
accs, cis_lo, cis_hi = [], [], []
for c in conditions:
    outcomes = [x["correct"] for x in eval_all[c]]
    mean, lo, hi = bootstrap_ci(outcomes)
    accs.append(mean)
    cis_lo.append(mean - lo)
    cis_hi.append(hi - mean)

colors = ["#1f77b4", "#ff7f0e", "#ff7f0e", "#2ca02c", "#2ca02c"]
hatches = ["", "", "//", "", "//"]
bars = axes[0].bar(range(len(conditions)), accs, yerr=[cis_lo, cis_hi],
                   capsize=5, color=colors, edgecolor="black")
for bar, h in zip(bars, hatches):
    bar.set_hatch(h)
axes[0].set_xticks(range(len(conditions)))
axes[0].set_xticklabels([LABELS[c] for c in conditions], rotation=30, ha="right")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Eval Accuracy (AIME 2025)")
axes[0].axhline(y=baseline_acc, color="gray", linestyle="--",
                label=f"Baseline: {baseline_acc:.1%}")
axes[0].legend()

# Fig 2: Absorption delta
c_acc = sum(x["correct"] for x in eval_all["C"]) / len(eval_all["C"])
cabs_acc = sum(x["correct"] for x in eval_all["C-abs"]) / len(eval_all["C-abs"])
absorption_data = {
    "Static (C->C-abs)": c_acc - cabs_acc,
    "Active (D->D-abs)": d_acc - dabs_acc,
}
axes[1].bar(absorption_data.keys(), absorption_data.values(),
            color=["#ff7f0e", "#2ca02c"], edgecolor="black")
axes[1].axhline(y=0, color="gray", linestyle="-")
axes[1].set_ylabel("Accuracy Drop When Playbook Removed")
axes[1].set_title("Absorption Test")

plt.tight_layout()
plt.savefig("pilot_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pilot_results.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fig 1: Training time per epoch
for label, metrics in [("B", b_metrics), ("C", c_metrics), ("D", d_metrics)]:
    epochs = [m["epoch"] for m in metrics]
    times = [m["time_s"] for m in metrics]
    axes[0].plot(epochs, times, "o-", label=label)

# Overlay playbook size for D
if d_metrics[0].get("pb_size", 0) > 0:
    ax2 = axes[0].twinx()
    ax2.plot([m["epoch"] for m in d_metrics],
             [m["pb_size"] for m in d_metrics],
             "s--", color="gray", alpha=0.5, label="PB size (D)")
    ax2.set_ylabel("Playbook size (D only)")
    ax2.legend(loc="upper left")

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Time (s)")
axes[0].set_title("Training Time per Epoch")
axes[0].legend(loc="upper right")

# Fig 2: Answer entropy (collapse detection)
for label, metrics in [("B", b_metrics), ("C", c_metrics), ("D", d_metrics)]:
    epochs = [m["epoch"] for m in metrics]
    entropies = [m.get("entropy", 0) for m in metrics]
    axes[1].plot(epochs, entropies, "o-", label=label)
axes[1].axhline(y=0.1, color="red", linestyle="--", alpha=0.5, label="Collapse threshold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Shannon Entropy")
axes[1].set_title("Answer Diversity (Collapse Detection)")
axes[1].legend()

plt.tight_layout()
plt.savefig("pilot_training.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pilot_training.png")

## 9. Verdict + Next Steps

In [ ]:
# Map results to what-if matrix from revised-research-plan.md
gap_db = d_acc - b_acc
absorption = d_acc - dabs_acc

print("=" * 60)
print("VERDICT (What-If Matrix)")
print("=" * 60)
if gap_db > 0.10 and absorption < 0.05:
    print("-> Outcome 4: ABSORPTION (D-abs ~ D > B). Strongest paper.")
elif gap_db > 0.10:
    print("-> Outcome 1: SYNERGY (D > B, playbook still needed).")
elif gap_db > 0.05:
    print("-> Outcome 5: PARTIAL ABSORPTION.")
elif abs(gap_db) < 0.05:
    print("-> Outcome 2 or 9: REDUNDANCY or NOISE. Check diversity diagnostic at scale.")
elif gap_db < -0.05:
    print("-> Outcome 3: INTERFERENCE (co-evolution hurts RL).")
print(f"\n  D-B gap: {gap_db:+.1%}, Absorption: {absorption:+.1%}")
print(f"\nNEXT: If pilot is stable, scale up via Modal:")
print(f"  modal run experiments/memory_rl/run_modal.py --condition all --seed -1")

## 10. Cleanup & Save

In [ ]:
# Save all results
with open("pilot_results.json", "w") as f:
    json.dump({
        "baseline_acc": baseline_acc,
        "eval_results": {k: v for k, v in eval_all.items()},
        "training_metrics": {"B": b_metrics, "C": c_metrics, "D": d_metrics},
        "collapse_history": {"B": b_collapse, "C": c_collapse, "D": d_collapse},
        "config": {
            "seed": SEED, "n_train": N_TRAIN, "n_epochs": N_GRPO_EPOCHS,
            "num_generations": NUM_GENERATIONS, "model": MODEL_NAME,
            "per_device_train_batch_size": NUM_GENERATIONS,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
        },
    }, f, indent=2, default=str)
print("Saved pilot_results.json")
log_gpu_memory("final")
print("\nDone! This pilot validates the full experiment pipeline.")
print("Scale up: modal run experiments/memory_rl/run_modal.py --condition all")